In [120]:
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, f1_score

from imblearn.over_sampling import RandomOverSampler
from imblearn.pipeline import Pipeline as ImbPipeline

ImportError: cannot import name '_fit_context' from 'sklearn.base' (/Users/feroj/opt/anaconda3/lib/python3.9/site-packages/sklearn/base.py)

In [76]:
df = pd.read_csv("risk-train.txt", sep="\t", na_values="?")

In [77]:
df = df.drop(columns=["ORDER_ID","Z_LAST_NAME", 'B_EMAIL', 'B_TELEFON'], errors="ignore")

In [78]:
numeric_cols = ['ANUMMER_02', 'ANUMMER_03', 'ANUMMER_04', 'ANUMMER_05', 'ANUMMER_06', 'ANUMMER_07', 'ANUMMER_08', 'ANUMMER_09', 'MAHN_AKT', 'MAHN_HOECHST']
df[numeric_cols] = df[numeric_cols].fillna(df[numeric_cols].median())

In [79]:
# Since ANUMMER_10 has NaN for all the row
df = df.drop(columns=['ANUMMER_10'], errors="ignore")

In [80]:
# Fill the birthdate and generate age
df["B_BIRTHDATE"] = pd.to_datetime(df["B_BIRTHDATE"], errors="coerce")
df["AGE"] = 2026 - df["B_BIRTHDATE"].dt.year
df.drop("B_BIRTHDATE", axis=1, inplace=True)

df["AGE"].fillna(df["AGE"].median(), inplace=True)

In [81]:
# Since categorical feature, we used the most frequent used category
df["Z_CARD_ART"].fillna(df["Z_CARD_ART"].mode()[0], inplace=True)

In [82]:
# Preprocessing Time_Order to hour and minutes so that it helps to use in our model
df["TIME_ORDER"] = pd.to_datetime(df["TIME_ORDER"], format="%H:%M", errors="coerce")
df["ORDER_HOUR"] = df["TIME_ORDER"].dt.hour
df["ORDER_MINUTE"] = df["TIME_ORDER"].dt.minute

df["ORDER_HOUR"].fillna(df["ORDER_HOUR"].median(), inplace=True)
df["ORDER_MINUTE"].fillna(df["ORDER_MINUTE"].median(), inplace=True)
df.drop("TIME_ORDER", axis=1, inplace=True)

In [83]:
# Preprocessing the date_lorder
df["DATE_LORDER"] = pd.to_datetime(df["DATE_LORDER"], format="%m/%d/%Y", errors="coerce")
df["LAST_ORDER_YEAR"] = df["DATE_LORDER"].dt.year
df["LAST_ORDER_MONTH"] = df["DATE_LORDER"].dt.month
df["LAST_ORDER_DAY"] = df["DATE_LORDER"].dt.day

df["LAST_ORDER_YEAR"].fillna(df["LAST_ORDER_YEAR"].median(), inplace=True)
df["LAST_ORDER_MONTH"].fillna(df["LAST_ORDER_MONTH"].median(), inplace=True)
df["LAST_ORDER_DAY"].fillna(df["LAST_ORDER_DAY"].median(), inplace=True)

df.drop("DATE_LORDER", axis=1, inplace=True)

In [87]:
# Check if the data is imbalanced
df["CLASS"].value_counts()

no     28254
yes     1746
Name: CLASS, dtype: int64

In [97]:
y = df["CLASS"].map({"no":0, "yes":1})

# Remove target and ID
X = df.drop(columns=["CLASS"])

In [108]:
# Convert the categorical column to numerical
X = pd.get_dummies(X, drop_first=True)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 30000 entries, 0 to 29999
Data columns (total 52 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   Z_CARD_VALID             30000 non-null  float64
 1   VALUE_ORDER              30000 non-null  float64
 2   AMOUNT_ORDER             30000 non-null  int64  
 3   ANUMMER_01               30000 non-null  int64  
 4   ANUMMER_02               30000 non-null  float64
 5   ANUMMER_03               30000 non-null  float64
 6   ANUMMER_04               30000 non-null  float64
 7   ANUMMER_05               30000 non-null  float64
 8   ANUMMER_06               30000 non-null  float64
 9   ANUMMER_07               30000 non-null  float64
 10  ANUMMER_08               30000 non-null  float64
 11  ANUMMER_09               30000 non-null  float64
 12  SESSION_TIME             30000 non-null  int64  
 13  AMOUNT_ORDER_PRE         30000 non-null  int64  
 14  VALUE_ORDER_PRE       

In [113]:
# Train-test split (Stratified)
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    stratify=y,
    random_state=42
)

In [107]:
X_train.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 24000 entries, 10416 to 3412
Data columns (total 43 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   B_EMAIL            24000 non-null  object 
 1   B_TELEFON          24000 non-null  object 
 2   FLAG_LRIDENTISCH   24000 non-null  object 
 3   FLAG_NEWSLETTER    24000 non-null  object 
 4   Z_METHODE          24000 non-null  object 
 5   Z_CARD_ART         24000 non-null  object 
 6   Z_CARD_VALID       24000 non-null  float64
 7   VALUE_ORDER        24000 non-null  float64
 8   WEEKDAY_ORDER      24000 non-null  object 
 9   AMOUNT_ORDER       24000 non-null  int64  
 10  ANUMMER_01         24000 non-null  int64  
 11  ANUMMER_02         24000 non-null  float64
 12  ANUMMER_03         24000 non-null  float64
 13  ANUMMER_04         24000 non-null  float64
 14  ANUMMER_05         24000 non-null  float64
 15  ANUMMER_06         24000 non-null  float64
 16  ANUMMER_07         

In [114]:
# Logistic Regression model
model = LogisticRegression(class_weight="balanced", max_iter=2000)

# Train using training data
model.fit(X_train, y_train)

LogisticRegression(class_weight='balanced', max_iter=2000)

In [116]:
pred = model.predict(X_test)

In [117]:
print(classification_report(y_test, pred))

              precision    recall  f1-score   support

           0       0.95      0.47      0.63      5651
           1       0.07      0.59      0.12       349

    accuracy                           0.48      6000
   macro avg       0.51      0.53      0.37      6000
weighted avg       0.90      0.48      0.60      6000



In [118]:
f1_yes = f1_score(y_test, pred, pos_label=1)
print("F1-score for YES class:", f1_yes)

F1-score for YES class: 0.11728045325779039


In [100]:
X_train.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 24000 entries, 10416 to 3412
Data columns (total 43 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   B_EMAIL            24000 non-null  object 
 1   B_TELEFON          24000 non-null  object 
 2   FLAG_LRIDENTISCH   24000 non-null  object 
 3   FLAG_NEWSLETTER    24000 non-null  object 
 4   Z_METHODE          24000 non-null  object 
 5   Z_CARD_ART         24000 non-null  object 
 6   Z_CARD_VALID       24000 non-null  float64
 7   VALUE_ORDER        24000 non-null  float64
 8   WEEKDAY_ORDER      24000 non-null  object 
 9   AMOUNT_ORDER       24000 non-null  int64  
 10  ANUMMER_01         24000 non-null  int64  
 11  ANUMMER_02         24000 non-null  float64
 12  ANUMMER_03         24000 non-null  float64
 13  ANUMMER_04         24000 non-null  float64
 14  ANUMMER_05         24000 non-null  float64
 15  ANUMMER_06         24000 non-null  float64
 16  ANUMMER_07         

In [24]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 30000 entries, 0 to 29999
Data columns (total 41 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   CLASS              30000 non-null  object 
 1   B_EMAIL            30000 non-null  object 
 2   B_TELEFON          30000 non-null  object 
 3   B_BIRTHDATE        27058 non-null  object 
 4   FLAG_LRIDENTISCH   30000 non-null  object 
 5   FLAG_NEWSLETTER    30000 non-null  object 
 6   Z_METHODE          30000 non-null  object 
 7   Z_CARD_ART         11346 non-null  object 
 8   Z_CARD_VALID       30000 non-null  float64
 9   VALUE_ORDER        30000 non-null  float64
 10  WEEKDAY_ORDER      30000 non-null  object 
 11  TIME_ORDER         29980 non-null  object 
 12  AMOUNT_ORDER       30000 non-null  int64  
 13  ANUMMER_01         30000 non-null  int64  
 14  ANUMMER_02         30000 non-null  float64
 15  ANUMMER_03         30000 non-null  float64
 16  ANUMMER_04         300